# Objective: compare runs, deploy to a REST API, and build a container image

In [10]:
import keras
import numpy as np
import pandas as pd
from hyperopt import fmin, tpe, hp, STATUS_OK, Trials
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split

import mlflow
from mlflow.models import infer_signature

In [11]:
data = pd.read_csv(
    "https://raw.githubusercontent.com/mlflow/mlflow/master/tests/datasets/winequality-white.csv",
    sep=";"
)
data

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
0,7.0,0.27,0.36,20.7,0.045,45.0,170.0,1.00100,3.00,0.45,8.8,6
1,6.3,0.30,0.34,1.6,0.049,14.0,132.0,0.99400,3.30,0.49,9.5,6
2,8.1,0.28,0.40,6.9,0.050,30.0,97.0,0.99510,3.26,0.44,10.1,6
3,7.2,0.23,0.32,8.5,0.058,47.0,186.0,0.99560,3.19,0.40,9.9,6
4,7.2,0.23,0.32,8.5,0.058,47.0,186.0,0.99560,3.19,0.40,9.9,6
...,...,...,...,...,...,...,...,...,...,...,...,...
4893,6.2,0.21,0.29,1.6,0.039,24.0,92.0,0.99114,3.27,0.50,11.2,6
4894,6.6,0.32,0.36,8.0,0.047,57.0,168.0,0.99490,3.15,0.46,9.6,5
4895,6.5,0.24,0.19,1.2,0.041,30.0,111.0,0.99254,2.99,0.46,9.4,6
4896,5.5,0.29,0.30,1.1,0.022,20.0,110.0,0.98869,3.34,0.38,12.8,7


In [12]:
# Train test split
X = data.drop(columns = ["quality"])
y = data["quality"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42)

# Format for ANN
train_x = X_train.values
train_y = y_train.values.ravel()
test_x = X_test.values
test_y = y_test.values.ravel()

# Validation
train_x, val_x, train_y, val_y = train_test_split(
    train_x, train_y, test_size = 0.2, random_state = 42
)

In [13]:
signature = infer_signature(train_x, train_y)
print(signature)

inputs: 
  [Tensor('float64', (-1, 11))]
outputs: 
  [Tensor('int64', (-1,))]
params: 
  None



## ANN model

In [14]:
def train_model(params, epochs, train_x, train_y, val_x, val_y, test_x, test_y):

    # Define model architecture
    mean = np.mean(train_x, axis=0)
    var = np.var(train_x, axis=0)
    model = keras.Sequential(
        [
            keras.Input([train_x.shape[1]]),
            keras.layers.Normalization(mean = mean, variance = var),
            keras.layers.Dense(64, activation = "relu"),
            keras.layers.Dense(1)
        ]
    )

    # Compile model
    model.compile(
        optimizer = keras.optimizers.SGD(
            learning_rate = params["lr"],
            momentum = params["momentum"]
        ),
        loss = "mean_squared_error",
        metrics = [keras.metrics.RootMeanSquaredError()]
    )

    # Train model with MLflow tracking
    with mlflow.start_run(nested = True):

        model.fit(
            train_x, train_y,
            validation_data = (val_x, val_y),
            epochs = epochs,
            batch_size = 64
        )

        # Evaluate model
        eval_results = model.evaluate(val_x, val_y, batch_size = 64)
        eval_rmse = eval_results[1]

        # Log the parameters and metrics to MLflow
        mlflow.log_params(params)
        mlflow.log_metric("val_rmse", eval_rmse)

        # Log the model
        mlflow.tensorflow.log_model(model, "model", signature = signature)

        return {"loss": eval_rmse, "status": STATUS_OK, "model": model}

In [15]:
def objective(params):
    return train_model(
        params, 
        epochs = 3,
        train_x = train_x, 
        train_y = train_y,
        val_x = val_x, 
        val_y = val_y,
        test_x = test_x, 
        test_y = test_y
    )

In [16]:
space = {
    "lr": hp.loguniform("lr", np.log(1e-5), np.log(1e-1)),
    "momentum": hp.uniform("momentum", 0.0, 1.0)
}

In [18]:
mlflow.set_experiment("/wine-quality")

with mlflow.start_run():

    # Conduct hyperparameter optimization
    trials = Trials()
    best = fmin(
        fn = objective,
        space = space,
        algo = tpe.suggest,
        max_evals = 4,
        trials = trials
    )

    # Fetch the details of the best run
    best_run = sorted(trials.results, key = lambda x: x["loss"])[0]

    # Log the best hyperparameters and metrics to MLflow
    mlflow.log_params(best)
    mlflow.log_metric("best_val_rmse", best_run["loss"])
    mlflow.tensorflow.log_model(best_run["model"], "best_model", signature = signature)

  0%|          | 0/4 [00:00<?, ?trial/s, best loss=?]WARNING:tensorflow:TensorFlow GPU support is not available on native Windows for TensorFlow >= 2.11. Even if CUDA/cuDNN are installed, GPU will not be used. Please use WSL2 or the TensorFlow-DirectML plugin.
Epoch 1/3                                            

 1/49 ━━━━━━━━━━━━━━━━━━━━ 1:05 1s/step - loss: 35.4480 - root_mean_squared_error: 5.9538
 9/49 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 30.5920 - root_mean_squared_error: 5.5249 
18/49 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 26.0525 - root_mean_squared_error: 5.0784
25/49 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 23.2426 - root_mean_squared_error: 4.7764
34/49 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 20.4391 - root_mean_squared_error: 4.4530
45/49 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 17.9022 - root_mean_squared_error: 4.1395
49/49 ━━━━━━━━━━━━━━━━━━━━ 2s 14ms/step - loss: 8.5416 - root_mean_squared_error: 2.9226 - val_loss: 2.1409 - val_root_mean_squared_error: 1.4632

E

2026/05/25 17:29:39 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.



Epoch 1/3                                                                     

 1/49 ━━━━━━━━━━━━━━━━━━━━ 34s 718ms/step - loss: 30.8247 - root_mean_squared_error: 5.5520
25/49 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 6.9080 - root_mean_squared_error: 2.4744    
48/49 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 4.6783 - root_mean_squared_error: 2.0070
49/49 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - loss: 1.8622 - root_mean_squared_error: 1.3646 - val_loss: 0.6343 - val_root_mean_squared_error: 0.7964

Epoch 2/3                                                                     

 1/49 ━━━━━━━━━━━━━━━━━━━━ 4s 92ms/step - loss: 0.4708 - root_mean_squared_error: 0.6861
 8/49 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.6299 - root_mean_squared_error: 0.7926 
15/49 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.6540 - root_mean_squared_error: 0.8080
22/49 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.6516 - root_mean_squared_error: 0.8067
29/49 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.6473 - root_mean_sq

2026/05/25 17:30:11 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.



Epoch 1/3                                                                      

 1/49 ━━━━━━━━━━━━━━━━━━━━ 48s 1s/step - loss: 36.9741 - root_mean_squared_error: 6.0806
 6/49 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 36.1939 - root_mean_squared_error: 6.0160
 8/49 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - loss: 36.0462 - root_mean_squared_error: 6.0037
 9/49 ━━━━━━━━━━━━━━━━━━━━ 1s 45ms/step - loss: 36.0177 - root_mean_squared_error: 6.0013
15/49 ━━━━━━━━━━━━━━━━━━━━ 1s 31ms/step - loss: 35.8193 - root_mean_squared_error: 5.9848
23/49 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - loss: 35.6427 - root_mean_squared_error: 5.9700
28/49 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - loss: 35.5235 - root_mean_squared_error: 5.9600
33/49 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - loss: 35.4180 - root_mean_squared_error: 5.9511
39/49 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - loss: 35.3136 - root_mean_squared_error: 5.9424
49/49 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - loss: 35.1752 - root_mean_squared_error: 5.9307
49/49 ━━━━━━━━━━━━━━

2026/05/25 17:30:31 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.



Epoch 1/3                                                                      

 1/49 ━━━━━━━━━━━━━━━━━━━━ 24s 518ms/step - loss: 37.2239 - root_mean_squared_error: 6.1011
31/49 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 19.7271 - root_mean_squared_error: 4.3458   
47/49 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 15.8461 - root_mean_squared_error: 3.8476
49/49 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - loss: 6.9897 - root_mean_squared_error: 2.6438 - val_loss: 1.9393 - val_root_mean_squared_error: 1.3926

Epoch 2/3                                                                      

 1/49 ━━━━━━━━━━━━━━━━━━━━ 3s 78ms/step - loss: 1.3694 - root_mean_squared_error: 1.1702
11/49 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 1.4248 - root_mean_squared_error: 1.1936 
19/49 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 1.4117 - root_mean_squared_error: 1.1881
29/49 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 1.4048 - root_mean_squared_error: 1.1852
39/49 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 1.4087 - root_mea

2026/05/25 17:30:52 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.



100%|██████████| 4/4 [01:39<00:00, 25.00s/trial, best loss: 0.7154267430305481]

2026/05/25 17:31:15 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Once we identify the best model, we assign it in the Model Registry in the UI

In [ ]:
# Alternative of registration: via code

mlflow.register_model(
    "runs:/<run_id>/best_model",
    "WineQualityModel"
)